This takes a Folder containing subFolders that contain the post processing data from each recording in a Folder called 'analyzer clean' 
retuns a csv with columns 1: Original_index: Subfoldername_originalRownumber which corresponds to the Phy_ID.
This is limited since there are new units created by manual curation which can never recieve the FS tag. There is also a version that uses Si-unit_id. not relying on the order in Phy but it results in the same results. 

 For Neuropixles recordings, all the required infos are contained in the Data 

In [ ]:
import os
import spikeinterface.full as si  

import pandas as pd  
import matplotlib.pyplot as plt
from pathlib import Path 

def find_key_in_dict(d, key_to_find):
    for key, value in d.items():
        if key == key_to_find:
            # If the value is a Pandas Series, convert it to a list
            if isinstance(value, pd.Series):
                return value.tolist()
            return value
        elif isinstance(value, dict):
            result = find_key_in_dict(value, key_to_find)
            if result:
                return result
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    result = find_key_in_dict(item, key_to_find)
                    if result:
                        return result
    return None

# Base directory where subfolders are located
base_dir = Path("E:/Florian_paper/Florian/Data/Opto_2")


# Initialize a list to store all peak-to-valley values across subfolders
all_peak_to_valley_ms = []
original_indices = []  # To store the original indices for each value

# Iterate through all subfolders
for subfolder in base_dir.iterdir():  # Using pathlib to iterate
    if subfolder.is_dir():
        try:
            # Construct the full path and normalize it
            analyzer_path = subfolder / "analyzer_clean"
            analyzer_path = os.path.normpath(analyzer_path)  # Normalize the path
            print(analyzer_path)
            # Load the analyzer for the current subfolder
            analyzer = si.load_sorting_analyzer(str(analyzer_path))  # Convert to string if needed
            tm = analyzer.get_extension('template_metrics')
            dat = tm.data

            # Extract peak-to-valley data
            peak_to_valley_data = find_key_in_dict(dat['metrics'], 'peak_to_valley')
            
            # Check if peak_to_valley_data is found
            if peak_to_valley_data:
                # If it's a Pandas Series, convert it to a list
                if isinstance(peak_to_valley_data, pd.Series):
                    peak_to_valley_data = peak_to_valley_data.tolist()
                
                # Convert to milliseconds and append to the cumulative list
                peak_to_valley_ms = [value * 1000 for value in peak_to_valley_data]
                
                # Append the values and their original indices
                all_peak_to_valley_ms.extend(peak_to_valley_ms)
                original_indices.extend([f"{subfolder.name}_row_{i}" for i in range(len(peak_to_valley_ms))])

                
        except Exception as e:
            print(f"Error processing {subfolder}: {e}")

# Filter values below 0.425ms  from the list
filtered_peak_to_valley_ms = [
    (index, value) for index, value in zip(original_indices, all_peak_to_valley_ms) if value < 0.425
]

# Create a DataFrame from the filtered data, including the original indices
df_filtered = pd.DataFrame(filtered_peak_to_valley_ms, columns=["Original_Index", "Peak_to_Valley (ms)"])

# Display the DataFrame
print(df_filtered)

#df_filtered.to_csv("C:/Users/Freitag/Desktop/Batch3_SIidx.csv")
print(analyzer.has_recording()) 

For cambridgeneurotech. Need to load recording and sorting speratly and recompute metrics since post-processing was done with the outdated waveforme extractor. adds Probe informations sice those are not in the metadata

In [ ]:
import os
import spikeinterface.full as si 
import pandas as pd  
import matplotlib.pyplot as plt
from pathlib import Path

def find_key_in_dict(d, key_to_find):
    for key, value in d.items():
        if key == key_to_find:
            # If the value is a Pandas Series, convert it to a list
            if isinstance(value, pd.Series):
                return value.tolist()
            return value
        elif isinstance(value, dict):
            result = find_key_in_dict(value, key_to_find)
            if result:
                return result
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    result = find_key_in_dict(item, key_to_find)
                    if result:
                        return result
    return None
base_dir = Path("E:/Florian/Data/batch1")
all_peak_to_valley_ms = []
original_indices = []  

for subfolder in base_dir.iterdir():  # Using pathlib to iterate
    if subfolder.is_dir():
        try:
            # Construct the path for sorting output (fixed)
            sorting_output_path = subfolder / "sorted/sorter_output"
            sorting_output_path = os.path.normpath(sorting_output_path)  # Normalize the path

            # Search for a subfolder containing 'Record Node' and a number
            record_node_folders = [f for f in subfolder.iterdir() if "Record Node" in f.name]
            if not record_node_folders:
                print(f"No 'Record Node' found in {subfolder}. Skipping...")
                continue
            record_node_folder = record_node_folders[0]
            record_node_path = os.path.normpath(record_node_folder)  # Normalize the path
            
            # Print paths for debugging
            print(f"Sorting Output Path: {sorting_output_path}")
            print(f"Record Node Path: {record_node_path}")

            # Load Sorting (Kilosort) data
            Sorting_KS4 = si.read_kilosort(str(sorting_output_path))  # Ensure path is passed as string

            # Load OpenEphys data
            rec1 = si.read_openephys(str(record_node_path))

            # Download probe
            probe = pi.get_probe(manufacturer='cambridgeneurotech', probe_name='ASSY-77-H3')
            
            # Add wiring to device
            probe.wiring_to_device('ASSY-77>Adpt.A64-Om32_2x-sm-NN>RHD2164')

            # Set the probe for the current recording
            rec1.set_probe(probe, group_mode="by_shank", in_place=True)

            # Create the sorting analyzer with the loaded data
            analyzer = si.create_sorting_analyzer(Sorting_KS4, rec1, sparse=True, format="memory")

            # Compute the necessary metrics
            global_job_kwargs = {}  # You can define the global arguments here if needed
            analyzer.compute(['random_spikes', 'waveforms', 'templates'], **global_job_kwargs)

            # Compute 'template_metrics'
            tm = analyzer.compute('template_metrics')
            dat = tm.data

            peak_to_valley_data = find_key_in_dict(dat['metrics'], 'peak_to_valley')
            
            # Check if peak_to_valley_data is found
            if peak_to_valley_data:
                # If it's a Pandas Series, convert it to a list
                if isinstance(peak_to_valley_data, pd.Series):
                    peak_to_valley_data = peak_to_valley_data.tolist()
                
                # Convert to milliseconds and append to the cumulative list
                peak_to_valley_ms = [value * 1000 for value in peak_to_valley_data]
                
                # Append the values and their original indices
                all_peak_to_valley_ms.extend(peak_to_valley_ms)
                original_indices.extend([f"{subfolder.name}_row_{i}" for i in range(len(peak_to_valley_ms))])
        
        except Exception as e:
                print(f"Error processing {subfolder}: {e}")

# Filter values below 0.425ms (450ms) from the list
filtered_peak_to_valley_ms = [
    (index, value) for index, value in zip(original_indices, all_peak_to_valley_ms) if value < 0.425
]

# Create a DataFrame from the filtered data, including the original indices
df_filtered = pd.DataFrame(filtered_peak_to_valley_ms, columns=["Original_Index", "Peak_to_Valley (ms)"])

# Display the DataFrame
print(df_filtered)

#df_filtered.to_csv("C:/Users/deepl/Desktop/Batch1.csv")





for using SI_unit_ids

In [ ]:
# import os
# import spikeinterface.full as si  
# import pandas as pd  
# import matplotlib.pyplot as plt
# from pathlib import Path 

# def find_key_in_dict(d, key_to_find):
#     for key, value in d.items():
#         if key == key_to_find:
#             if isinstance(value, pd.Series):
#                 return value.tolist()
#             return value
#         elif isinstance(value, dict):
#             result = find_key_in_dict(value, key_to_find)
#             if result:
#                 return result
#         elif isinstance(value, list):
#             for item in value:
#                 if isinstance(item, dict):
#                     result = find_key_in_dict(item, key_to_find)
#                     if result:
#                         return result
#     return None

# # Base directory
# base_dir = Path("E:/Florian_paper/Florian/Data/batch3")

# all_peak_to_valley_ms = []
# original_indices = []

# for subfolder in base_dir.iterdir():
#     if subfolder.is_dir():
#         try:
#             analyzer_path = subfolder / "analyzer_clean"
#             analyzer_path = os.path.normpath(analyzer_path)
#             print(analyzer_path)
            
#             analyzer = si.load_sorting_analyzer(str(analyzer_path))
#             tm = analyzer.get_extension('template_metrics')
#             dat = tm.data

#             peak_to_valley_data = find_key_in_dict(dat['metrics'], 'peak_to_valley')
            
#             if peak_to_valley_data:
#                 if isinstance(peak_to_valley_data, pd.Series):
#                     peak_to_valley_data = peak_to_valley_data.tolist()
                
#                 peak_to_valley_ms = [value * 1000 for value in peak_to_valley_data]
#                 unit_ids = analyzer.sorting.unit_ids

#                 for i, value in enumerate(peak_to_valley_ms):
#                     original_indices.append(f"{subfolder.name}_unit_{unit_ids[i]}")
#                     all_peak_to_valley_ms.append(value)

#         except Exception as e:
#             print(f"Error processing {subfolder}: {e}")

# # Filter
# filtered_peak_to_valley_ms = [
#     (index, value) for index, value in zip(original_indices, all_peak_to_valley_ms) if value < 0.425
# ]

# df_filtered = pd.DataFrame(filtered_peak_to_valley_ms, columns=["Original_Index", "Peak_to_Valley (ms)"])
# print(df_filtered)

# df_filtered.to_csv("C:/Users/Freitag/Desktop/Batch3_SIidx.csv")
